# Analyser ses données trail (1/3) — Lire les données de sa montre

Notebook associé au billet : https://gregs1t.github.io/trail-lab/2026-04-01-trail-analyse-1-lire-donnees/

**Ce qu'on fait ici :**
- Charger un fichier `.fit` dans un DataFrame pandas
- Nettoyage minimal (altitude, distance, allure)
- Résumé chiffré (distance, durée, D+/D-, FC)
- Profil altimétrique avec ravitaillements
- Profils colorés par FC, allure, température
- Tableau des zones de fréquence cardiaque

---

**Pré-requis :**
```bash
pip install fitparse pandas numpy matplotlib
```
---

**Utilisation**

- C'est avant tout un notebook exploratoire. 
- Je te conseille d'en garder une copie propre quelque part si jamais tu veux revenir sur toutes les modifications que tu auras faites. 

- Sinon, tu changes le nom de ta trace FIT, tu ajustes les ravitos s'il y en a.

- Ensuite, tu cliques sur "Run All" et tu laisses dérouler l'execution.

**Remarque**

Dans les premiers notebooks, j'ai volontairement laissé le code complet dans les cellules. Ca alourdi un peu le contenu mais permet de voir directement le code associé. La bonne pratique est de mettre l'essentiel des fonctions dans un fichier associé, une librairie.

### 
### © Gregory Sainton
### 
- Notebook        : Analyse d'une séance de trail/course 
- Auteur          : Grégory Sainton <twinity4trail@proton.me>
- Dépôt           : https://github.com/gregs1t/trail-lab
- Date de création: 2025-12
- Dernière modif. : 2026-04
- Version         : 1.0
---
- Licence         : CC BY-NC-SA 4.0
-                   https://creativecommons.org/licenses/by-nc-sa/4.0/

>    Vous êtes libre de partager et d'adapter ce travail, à condition de :
>     · citer l'auteur (BY)
>     · ne pas en faire un usage commercial (NC)
>     · redistribuer sous la même licence (SA)

## Paramètres — à modifier avant de lancer

C'est la seule cellule à adapter à ta course.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fitparse import FitFile

pd.set_option("display.max_columns", 30)

# --- À adapter à ta course ou ton entrainement ---
FIT_PATH = "mon_fichier_a_moi.fit"   # chemin vers ton fichier .fit

FC_MAX = 185     # ta FC max réelle (bpm)
FC_MIN = 47      # ta FC repos (bpm)

# Laisse vide si pas de ravitaillements : RAVITO_KM = []
RAVITO_KM  = [19.2, 34.0, 45.0, 58.8, 65.4]
RAVITO_NOM = ["St Christo", "Ste Catherine", "St Genou", "Soucieu", "Chaponost"]


# --- Fin de la partie à adapter ---

## 1. Chargement du fichier FIT

Un fichier `.fit` contient des messages de types variés. On lit uniquement les messages `record` — un point par seconde environ — qui contiennent altitude, FC, vitesse, etc.

In [ ]:
def load_fit(fit_path):
    """Load FIT file records into a pandas DataFrame."""
    fitfile = FitFile(fit_path)
    records = []
    for record in fitfile.get_messages("record"):
        row = {}
        for field in record:
            row[field.name] = field.value
        records.append(row)
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["time_h"] = (
        (df["timestamp"] - df["timestamp"].iloc[0])
        .dt.total_seconds() / 3600.0
    )
    return df


df = load_fit(FIT_PATH)
print(f"{len(df)} points enregistrés")
print(f"Colonnes disponibles : {df.columns.tolist()}")

## 2. Nettoyage minimal

Deux points importants :
- On choisit `enhanced_altitude` si disponible (altimètre baro, plus précis que le GPS).
- On force la distance à ne jamais décroître (`np.maximum.accumulate`) pour éliminer les micro-reculs GPS.

In [ ]:
def clean_df(df):
    """Select best columns, enforce monotone distance, compute pace."""
    alt_col = "enhanced_altitude" if "enhanced_altitude" in df.columns else "altitude"
    spd_col = "enhanced_speed" if "enhanced_speed" in df.columns else "speed"

    df = df.dropna(subset=["distance", alt_col]).reset_index(drop=True)

    df["dist_m"] = np.maximum.accumulate(df["distance"].to_numpy(dtype=float))
    df["alt_m"] = df[alt_col].to_numpy(dtype=float)

    if spd_col in df.columns:
        v = df[spd_col].to_numpy(dtype=float)
        df["speed_mps"] = v
        df["speed_kmh"] = v * 3.6
        df["pace_s_per_km"] = np.where(v > 0.5, 1000.0 / v, np.nan)
    else:
        df["speed_mps"] = np.nan
        df["speed_kmh"] = np.nan
        df["pace_s_per_km"] = np.nan

    return df


df = clean_df(df)
df[["dist_m", "alt_m", "speed_kmh", "pace_s_per_km"]].describe().round(1)

## 3. Résumé chiffré

Le D+ est **filtré** : lissage de l'altitude + seuil minimal de variation pour éliminer le bruit GPS. Sans ce filtrage, le dénivelé peut être surestimé de 15 à 30 % selon la montre.

In [ ]:
def compute_dplus_dminus(df, dz_thr=None):
    """Compute filtered elevation gain and loss.

    dz_thr : minimum variation (m) to count. If None, calibrated
             automatically to raw_std/4 — adapts to both integer-resolution
             (Garmin entry-level) and high-resolution altimeters.
    """
    alt_smooth = (
        df["alt_m"]
        .rolling(7, center=True, min_periods=1).median()
        .rolling(7, center=True, min_periods=1).mean()
    )
    dz = alt_smooth.diff()
    if dz_thr is None:
        raw_std = df["alt_m"].diff().std()
        dz_thr = max(0.1, raw_std / 4.0)
    return float(dz[dz > dz_thr].sum()), float((-dz[dz < -dz_thr]).sum())


dplus, dminus = compute_dplus_dminus(df)

print(f"Distance  : {df['dist_m'].max() / 1000:.1f} km")
print(f"Durée     : {df['time_h'].max():.2f} h")
print(f"D+ filtré : {dplus:.0f} m")
print(f"D- filtré : {dminus:.0f} m")

if "heart_rate" in df.columns:
    hr = df["heart_rate"].dropna()
    print(f"FC moy.   : {hr.mean():.0f} bpm")
    print(f"FC max    : {hr.max():.0f} bpm")

## 4. Profil altimétrique avec ravitaillements

In [ ]:
def plot_profil(df, ravito_km, ravito_nom):
    """Plot elevation profile with aid station markers."""
    x = df["dist_m"] / 1000.0
    y = df["alt_m"]

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(x, y, color="brown", linewidth=2)
    ax.fill_between(x, y, y.min(), color="brown", alpha=0.3)

    for km, nom in zip(ravito_km, ravito_nom):
        ax.axvline(x=km, color="black", linestyle="--", alpha=0.6, linewidth=1)
        ax.text(km, y.max() + 20, nom,
                rotation=90, va="bottom", ha="center", fontsize=9)

    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("Altitude (m)")
    ax.set_title("Profil altimétrique")
    ax.grid(True)
    fig.tight_layout()
    plt.show()


plot_profil(df, RAVITO_KM, RAVITO_NOM)

## 5. Profils colorés

On colore chaque point du profil selon la variable d'intérêt. Avantage par rapport à un graphique temporel : on voit *où sur le parcours* quelque chose se passe.

Une seule fonction, appelée plusieurs fois avec des paramètres différents.

In [ ]:
def plot_profil_colore(df, col, label, cmap="viridis", vmin=None, vmax=None):
    """Plot elevation profile with scatter colored by a data column."""
    mask = df[col].notna()
    x = df.loc[mask, "dist_m"] / 1000.0
    y = df.loc[mask, "alt_m"]
    c = df.loc[mask, col]

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(
        df["dist_m"] / 1000.0, df["alt_m"], df["alt_m"].min(),
        color="lightgrey", alpha=0.5, zorder=1
    )
    sc = ax.scatter(x, y, c=c, cmap=cmap, s=4, alpha=0.8,
                    vmin=vmin, vmax=vmax, zorder=2)
    plt.colorbar(sc, ax=ax, label=label)
    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("Altitude (m)")
    ax.set_title(f"Profil — {label}")
    ax.grid(True)
    fig.tight_layout()
    plt.show()


print("Fonction plot_profil_colore définie.")

In [ ]:
# FC — coolwarm : bleu = zones basses, rouge = zones hautes
# vmin/vmax calés sur FC repos/max pour pouvoir comparer plusieurs courses
if "heart_rate" in df.columns:
    plot_profil_colore(df, "heart_rate", "FC (bpm)",
                       cmap="coolwarm", vmin=FC_MIN, vmax=FC_MAX)
else:
    print("Pas de données FC dans ce fichier.")

In [ ]:
# Allure — RdYlGn_r : rouge = lent, vert = rapide
# Le _r inverse la colormap (sans lui, vert = grandes valeurs = lent, contre-intuitif)
# Bornes 180 s/km (~5:30/km) et 900 s/km (~15:00/km) — à adapter à ta course
if "pace_s_per_km" in df.columns:
    plot_profil_colore(df, "pace_s_per_km", "Allure (s/km)",
                       cmap="RdYlGn_r", vmin=180, vmax=900)
else:
    print("Pas de données d'allure.")

In [ ]:
# Température — valeur biaisée à la hausse (poignet + ambiance)
# Utile pour observer les variations spatiales (vallée froide vs crête exposée)
if "temperature" in df.columns:
    plot_profil_colore(df, "temperature", "Température (°C)", cmap="plasma")
else:
    print("Pas de données de température dans ce fichier.")

## 6. Dashboard — toutes les variables alignées

`sharex=True` : zoomer sur un tronçon dans un panel déplace tous les autres automatiquement.

In [ ]:
def plot_dashboard(df, fc_min, fc_max):
    """Multi-panel elevation profiles colored by HR, pace, and temperature."""
    variables = [
        ("heart_rate",    "FC (bpm)",         "coolwarm",  fc_min, fc_max),
        ("pace_s_per_km", "Allure (s/km)",    "RdYlGn_r",  180,   900),
        ("temperature",   "Température (°C)", "plasma",    None,  None),
    ]
    variables = [(c, l, cm, vn, vx) for c, l, cm, vn, vx in variables
                 if c in df.columns]
    n = len(variables)
    if n == 0:
        print("Aucune variable disponible.")
        return

    fig, axes = plt.subplots(n, 1, figsize=(13, 4 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, (col, label, cmap, vmin, vmax) in zip(axes, variables):
        mask = df[col].notna()
        x = df.loc[mask, "dist_m"] / 1000.0
        y = df.loc[mask, "alt_m"]
        c = df.loc[mask, col]
        ax.fill_between(
            df["dist_m"] / 1000.0, df["alt_m"], df["alt_m"].min(),
            color="lightgrey", alpha=0.5, zorder=1
        )
        sc = ax.scatter(x, y, c=c, cmap=cmap, s=4, alpha=0.8,
                        vmin=vmin, vmax=vmax, zorder=2)
        plt.colorbar(sc, ax=ax, label=label)
        ax.set_ylabel("Altitude (m)")
        ax.set_title(f"Profil — {label}")
        ax.grid(True)

    axes[-1].set_xlabel("Distance (km)")
    fig.tight_layout()
    plt.show()


plot_dashboard(df, FC_MIN, FC_MAX)

## 7. Zones de fréquence cardiaque

In [ ]:
def compute_hr_zones(df, fc_max, hr_col="heart_rate"):
    """Compute time in HR zones based on %FCmax."""
    zones = [
        (0.50, 0.60, "Z1 — Récupération"),
        (0.60, 0.70, "Z2 — Endurance fond."),
        (0.70, 0.80, "Z3 — Tempo"),
        (0.80, 0.90, "Z4 — Seuil"),
        (0.90, 1.00, "Z5 — VO₂max"),
    ]
    df = df.copy()
    df["hr_frac"] = df[hr_col] / fc_max
    n_total = df[hr_col].notna().sum()
    rows = []
    for low, high, label in zones:
        mask = (df["hr_frac"] >= low) & (df["hr_frac"] < high)
        t_s = mask.sum()
        rows.append({
            "Zone": label,
            "FC min (bpm)": int(low * fc_max),
            "FC max (bpm)": int(high * fc_max),
            "Temps (min)": round(t_s / 60, 1),
            "Temps (%)": round(t_s / n_total * 100, 1),
        })
    return pd.DataFrame(rows)


if "heart_rate" in df.columns:
    print(compute_hr_zones(df, FC_MAX).to_string(index=False))
else:
    print("Pas de données FC.")

---
**Suite** : notebook `02_terrain.ipynb` — pente robuste, segmentation montées/descentes, GAP, VAM.